# 5-Fold Cross-Validation и OOF Blending

**Методология:**
1. Обучающая выборка (`train`) разбивается на одни и те же 5 стратифицированных фолдов (`StratifiedKFold`).
2. Каждая модель делает Out-Of-Fold (OOF) предсказания — каждый объект предсказывается моделью, которая не видела его при обучении.
3. Считаем `mean` и `std` ROC-AUC по 5 фолдам для каждой отдельной модели.
4. Ансамбли (бленды) оцениваем **только** по OOF предсказаниям фолд-за-фолдом.
5. Валидационная выборка больше не используется для выбора гиперпараметров и бленда во избежание data leakage / оверфита на holdout.


### 1. Данные и генерация признаков (Raw + Pair)


In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

SEED = 20260916

DATA_DIR = next(p for p in [".", "..", "../.."] if os.path.exists(f"{p}/train.csv"))

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feature_columns = [
    c for c in train.columns
    if c not in ["row_id", "target"]
]

X_raw = train[feature_columns].to_numpy()
X_test_raw = test[feature_columns].to_numpy()

y = train["target"].to_numpy()


def make_pair_features(X):
    A = X[:, :256]
    B = X[:, 256:]

    abs_diff = np.abs(A - B)

    return np.hstack([
        X,
        abs_diff,
    ])


X_pair = make_pair_features(X_raw)
X_test_pair = make_pair_features(X_test_raw)

feature_sets = {
    "raw": X_raw,
    "pair": X_pair,
}

test_feature_sets = {
    "raw": X_test_raw,
    "pair": X_test_pair,
}

print("raw:", X_raw.shape)
print("pair:", X_pair.shape)


### 2. Фиксированные 5 folds для всех моделей


In [ ]:
cv = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    ).split(X_raw, y)
)

fold_id = np.zeros(len(y), dtype=int)

for fold, (_, val_idx) in enumerate(cv):
    fold_id[val_idx] = fold


### 3. Спецификации всех исследуемых моделей


In [ ]:
model_specs = [
    {
        "name": "Decision Tree",
        "features": "raw",
        "factory": lambda: DecisionTreeClassifier(
            criterion="entropy",
            max_depth=10,
            max_features=0.3,
            min_samples_leaf=29,
            min_samples_split=2,
            splitter="best",
            ccp_alpha=0.005,
            random_state=SEED,
        ),
    },

    {
        "name": "Random Forest",
        "features": "raw",
        "factory": lambda: RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    {
        "name": "Gradient Boosting",
        "features": "raw",
        "factory": lambda: GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=SEED,
        ),
    },

    {
        "name": "Extra Trees raw",
        "features": "raw",
        "factory": lambda: ExtraTreesClassifier(
            n_estimators=100,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    {
        "name": "Extra Trees pair",
        "features": "pair",
        "factory": lambda: ExtraTreesClassifier(
            n_estimators=100,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    {
        "name": "Logistic raw",
        "features": "raw",
        "factory": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.003,
                solver="liblinear",
                max_iter=5000,
                random_state=SEED,
            )),
        ]),
    },

    {
        "name": "Logistic pair",
        "features": "pair",
        "factory": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.003,
                solver="liblinear",
                max_iter=5000,
                random_state=SEED,
            )),
        ]),
    },

    {
        "name": "XGBoost raw",
        "features": "raw",
        "factory": lambda: XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.15,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1,
            random_state=SEED,
            n_jobs=-1,
            eval_metric="auc",
        ),
    },

    {
        "name": "XGBoost pair",
        "features": "pair",
        "factory": lambda: XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.15,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1,
            random_state=SEED,
            n_jobs=-1,
            eval_metric="auc",
        ),
    },

    {
        "name": "CatBoost raw",
        "features": "raw",
        "factory": lambda: CatBoostClassifier(
            iterations=2000,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3,
            random_strength=2,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        ),
    },

    {
        "name": "CatBoost pair",
        "features": "pair",
        "factory": lambda: CatBoostClassifier(
            iterations=2000,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3,
            random_strength=2,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        ),
    },

    {
        "name": "LightGBM raw",
        "features": "raw",
        "factory": lambda: LGBMClassifier(
            boosting_type="goss",
            colsample_bytree=0.7,
            learning_rate=0.05,
            max_depth=5,
            min_child_samples=20,
            n_estimators=500,
            num_leaves=31,
            reg_lambda=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    },

    {
        "name": "LightGBM pair",
        "features": "pair",
        "factory": lambda: LGBMClassifier(
            boosting_type="goss",
            colsample_bytree=0.7,
            learning_rate=0.05,
            max_depth=5,
            min_child_samples=20,
            n_estimators=500,
            num_leaves=31,
            reg_lambda=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
]


### 4. Вычисление Out-Of-Fold (OOF) предсказаний


In [ ]:
oof_predictions = {}

cv_results = []

for model_number, spec in enumerate(model_specs, start=1):
    name = spec["name"]
    feature_name = spec["features"]

    X = feature_sets[feature_name]

    oof = np.zeros(len(y))

    fold_scores = []

    start = time.perf_counter()

    print(f"\n[{model_number}/{len(model_specs)}] {name}")

    for fold, (train_idx, val_idx) in enumerate(cv, start=1):
        model = spec["factory"]()

        model.fit(
            X[train_idx],
            y[train_idx],
        )

        prediction = model.predict_proba(
            X[val_idx]
        )[:, 1]

        oof[val_idx] = prediction

        auc = roc_auc_score(
            y[val_idx],
            prediction,
        )

        fold_scores.append(auc)

        print(f"fold {fold}: {auc:.6f}")

    elapsed = time.perf_counter() - start

    oof_predictions[name] = oof

    cv_results.append({
        "model": name,
        "mean_auc": np.mean(fold_scores),
        "std_auc": np.std(fold_scores),
        "fold_1": fold_scores[0],
        "fold_2": fold_scores[1],
        "fold_3": fold_scores[2],
        "fold_4": fold_scores[3],
        "fold_5": fold_scores[4],
        "time_sec": elapsed,
    })


### 5. Сводная таблица результатов всех моделей на 5-Fold CV


In [ ]:
cv_results_df = (
    pd.DataFrame(cv_results)
    .sort_values(
        "mean_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

cv_results_df


### 6. Функция оценки блендов (фолд-за-фолдом)


In [ ]:
def evaluate_blend(prediction):
    fold_scores = []

    for fold in range(5):
        mask = fold_id == fold

        score = roc_auc_score(
            y[mask],
            prediction[mask],
        )

        fold_scores.append(score)

    return {
        "mean_auc": np.mean(fold_scores),
        "std_auc": np.std(fold_scores),
        "fold_scores": fold_scores,
    }


### 7. Ранжирование внутри каждого фолда (Rank Averaging)


In [ ]:
def rank_oof(prediction):
    ranked = np.zeros_like(
        prediction,
        dtype=float,
    )

    for fold in range(5):
        mask = fold_id == fold

        ranked[mask] = (
            pd.Series(prediction[mask])
            .rank(
                method="average",
                pct=True,
            )
            .to_numpy()
        )

    return ranked


### 8. Отбор сильных моделей для блендинга


In [ ]:
strong_models = (
    cv_results_df[
        cv_results_df["mean_auc"] >= 0.88
    ]["model"]
    .tolist()
)

strong_models


### 9. Перебор комбинаций моделей (Probability Averaging + Rank Averaging)


In [ ]:
import itertools

blend_results = []

for size in range(
    2,
    len(strong_models) + 1,
):

    for combination in itertools.combinations(
        strong_models,
        size,
    ):

        # Probability averaging
        probability_prediction = np.mean(
            [
                oof_predictions[name]
                for name in combination
            ],
            axis=0,
        )

        probability_result = evaluate_blend(
            probability_prediction
        )

        blend_results.append({
            "method": "probability",
            "models": combination,
            "n_models": size,
            "mean_auc": probability_result["mean_auc"],
            "std_auc": probability_result["std_auc"],
        })


        # Rank averaging
        rank_prediction = np.mean(
            [
                rank_oof(
                    oof_predictions[name]
                )
                for name in combination
            ],
            axis=0,
        )

        rank_result = evaluate_blend(
            rank_prediction
        )

        blend_results.append({
            "method": "rank",
            "models": combination,
            "n_models": size,
            "mean_auc": rank_result["mean_auc"],
            "std_auc": rank_result["std_auc"],
        })


### 10. Настоящий рейтинг блендов (Mean CV ROC-AUC ± Std)


In [ ]:
blend_cv_df = (
    pd.DataFrame(blend_results)
    .sort_values(
        "mean_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

blend_cv_df.head(20)
